In [ ]:
!pip install transformers seqeval evaluate accelerate -U
!pip install transformers seqeval evaluate accelerate pytorch-crf -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c730ebd72bcb834b712502376af6094ed19a7bcf78b70ae975af1fc90201d921
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'
train_path = '/content/drive/MyDrive/datasetViMedNER/traindata/train.txt'
dev_path = '/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt'

unique_tags = []
with open(label_path, "r", encoding = "utf-8") as f :
    for line in f:
        line.strip()
        if line.strip():
          unique_tags.append(line.strip())

label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

train_sentences = load_conll_data(train_path)
dev_sentences = load_conll_data(dev_path)

In [ ]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from IPython.display import display

seqeval = evaluate.load("seqeval")

def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    # Rút trích riêng điểm của nguyen_nhan_benh
    f1_nguyen_nhan = 0.0
    if "nguyen_nhan_benh" in results:
        f1_nguyen_nhan = results["nguyen_nhan_benh"]["f1"]

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
        "f1_NNB": f1_nguyen_nhan  # Thêm cột F1 NNB vào log
    }

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
import pickle
train_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/train_dataset.pkl'
dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'
test_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/test_dataset.pkl'

def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset


train_dataset = load_dataset(train_dataset_filepath)
dev_dataset = load_dataset(dev_dataset_filepath)
test_dataset = load_dataset(test_dataset_filepath)

In [ ]:
sentence_lengths = []
entity_lengths = []
all_tags = set()
tag_counts = {}

for sent in train_sentences:
    sentence_lengths.append(len(sent))

    # Trích xuất thực thể từ chuỗi B- / I-
    in_entity = False
    current_ent_len = 0

    for word, tag in sent:
        all_tags.add(tag)
        tag_counts[tag] = tag_counts.get(tag, 0) + 1

        if tag.startswith('B-'):
            if in_entity:
                entity_lengths.append(current_ent_len)
            in_entity = True
            current_ent_len = 1
        elif tag.startswith('I-') and in_entity:
            current_ent_len += 1
        else:
            if in_entity:
                entity_lengths.append(current_ent_len)
                in_entity = False
                current_ent_len = 0
    if in_entity:
        entity_lengths.append(current_ent_len)

In [ ]:
label2id

{'B-bien_phap_chan_doan': 0,
 'B-bien_phap_dieu_tri': 1,
 'B-nguyen_nhan_benh': 2,
 'B-ten_benh': 3,
 'B-trieu_chung_benh': 4,
 'I-bien_phap_chan_doan': 5,
 'I-bien_phap_dieu_tri': 6,
 'I-nguyen_nhan_benh': 7,
 'I-ten_benh': 8,
 'I-trieu_chung_benh': 9,
 'O': 10}

In [ ]:
import math
import torch
import numpy as np

# Giả định Thành đã chạy sẵn tag_counts và label2id ở trên
sum_tag = sum(tag_counts.values())

# 1. Tính toán w_freq (Tần suất) và w_diff (Độ khó)
w_freq = np.zeros(len(label2id))
for tag, idx in label2id.items():
    w_freq[idx] = math.sqrt(sum_tag / tag_counts[tag])

f1_scores = {
    0: 0.7331, 1: 0.7152, 2: 0.2143, 3: 0.8694, 4: 0.6938,
    5: 0.6971, 6: 0.6108, 7: 0.1908, 8: 0.8946, 9: 0.6224,
    10: 0.9437 # Nhãn O
}

# Lấy điểm F1 cao nhất của nhóm thực thể (loại nhãn O)
f1_max = max(list(f1_scores.values())[:-1])

w_diff = np.zeros(len(label2id))
for tag, idx in label2id.items():
    w_diff[idx] = math.sqrt(f1_max / f1_scores[idx])

# 2. Tính static weight thô
static_weight_raw = w_freq * w_diff

# 3. Tinh chỉnh Scale (Chỉ xét min/max trên nhóm thực thể)
O_index = label2id['O']
entity_weights = np.delete(static_weight_raw, O_index)

min_w = np.min(entity_weights)
max_w = np.max(entity_weights)

target_min = 0.5
target_max = 3.0

static_weight_scaled = np.zeros_like(static_weight_raw)

for idx in range(len(static_weight_raw)):
    if idx == O_index:
        continue
    # Công thức Min-Max chuẩn: giữ nguyên tỷ lệ thuận (khó -> to, dễ -> nhỏ)
    static_weight_scaled[idx] = target_min + (static_weight_raw[idx] - min_w) * (target_max - target_min) / (max_w - min_w)

# 4. Ép cứng nhãn O xuống đáy (0.1)
static_weight_scaled[O_index] = 0.1

print("\n--- BẢNG TRỌNG SỐ SAU KHI SCALE CHUẨN ---")
for tag, idx in label2id.items():
    print(f"{idx:2d} | {tag:25s} | Weight = {static_weight_scaled[idx]:.4f}")

# 5. Chuyển thành PyTorch Tensor sẵn sàng dùng cho model
static_weight_tensor = torch.tensor(static_weight_scaled, dtype=torch.float)


--- BẢNG TRỌNG SỐ SAU KHI SCALE CHUẨN ---
 0 | B-bien_phap_chan_doan     | Weight = 1.6764
 1 | B-bien_phap_dieu_tri      | Weight = 1.2559
 2 | B-nguyen_nhan_benh        | Weight = 3.0000
 3 | B-ten_benh                | Weight = 0.7113
 4 | B-trieu_chung_benh        | Weight = 1.1118
 5 | I-bien_phap_chan_doan     | Weight = 1.0674
 6 | I-bien_phap_dieu_tri      | Weight = 0.8675
 7 | I-nguyen_nhan_benh        | Weight = 1.6380
 8 | I-ten_benh                | Weight = 0.5000
 9 | I-trieu_chung_benh        | Weight = 0.8562
10 | O                         | Weight = 0.1000


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF

class EntityAware_ViMedNER(nn.Module):
    def __init__(self, model_checkpoint, num_labels, static_weights, lambda_weight= 1.25, gamma_len=1.0):
        super().__init__()
        self.num_labels = num_labels
        self.lambda_weight = lambda_weight
        self.gamma_len = gamma_len
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.register_buffer('w_class', torch.tensor(static_weights, dtype=torch.float))

        # Thêm biến đếm để in log Loss
        self.batch_counter = 0

    def _get_span_weight_mask(self, labels):
        batch_size, seq_len = labels.shape
        mask = torch.ones((batch_size, seq_len), device=labels.device)

        O_LABEL_ID = 10
        # Khai báo danh sách các nhãn B- (dựa trên label2id)
        B_LABEL_IDS = [0, 1, 2, 3, 4]

        for i in range(batch_size):
            seq_labels = labels[i].tolist()
            start = -1
            for j in range(seq_len):
                lbl = seq_labels[j]

                # LOGIC MỚI: Cắt cụm nếu gặp O, PAD hoặc bắt đầu một nhãn B- mới
                if lbl == -100 or lbl == O_LABEL_ID or lbl in B_LABEL_IDS:
                    if start != -1:
                        # Kết thúc cụm cũ, tính w_len và gán
                        span_len = j - start
                        w_len = 1.0 + self.gamma_len * math.log(1.0 + span_len)
                        mask[i, start:j] = w_len
                        start = -1

                # Khởi tạo cụm mới nếu token là thực thể (B- hoặc I-)
                if lbl != -100 and lbl != O_LABEL_ID:
                    if start == -1:
                        start = j

            # Chốt sổ đoạn cuối câu
            if start != -1:
                span_len = seq_len - start
                w_len = 1.0 + self.gamma_len * math.log(1.0 + span_len)
                mask[i, start:seq_len] = w_len
        return mask

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        emissions = self.classifier(sequence_output)

        crf_mask = attention_mask.bool()
        crf_mask[:, 0] = True
        loss = None

        if labels is not None:
            # 1. TÍNH MAIN LOSS: CRF
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 10
            l_crf = -self.crf(emissions, safe_labels, mask=crf_mask, reduction='mean')

            # 2. TÍNH AUXILIARY LOSS: Entity-Aware
            l_aux_raw = F.cross_entropy(
                emissions.view(-1, self.num_labels),
                labels.view(-1),
                weight=self.w_class,
                ignore_index=-100,
                reduction='none'
            )

            w_len_mask = self._get_span_weight_mask(labels).view(-1)
            valid_tokens_mask = (labels != -100).view(-1)
            l_aux_weighted = (l_aux_raw * w_len_mask).sum() / (valid_tokens_mask.sum() + 1e-9)

            # 3. TỔNG LOSS
            loss = l_crf + (self.lambda_weight * l_aux_weighted)

        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(emissions, mask=crf_mask_decode)
        fake_logits = torch.zeros_like(emissions)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

In [ ]:
from torch.optim import AdamW
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
# 1. Khởi tạo mô hình Entity-Aware CRF mới của Thành
model_entity_aware = EntityAware_ViMedNER(
    model_checkpoint="vinai/phobert-base-v2",
    num_labels=len(label2id),
    static_weights=static_weight_tensor, # Sử dụng tensor trọng số đã tính ở bước trước
    lambda_weight=1.0,
    gamma_len=1.0
)

# 2. Tách nhóm tham số và gán Learning Rate riêng biệt (BÍ KÍP TỐI ƯU)
crf_params = list(model_entity_aware.crf.parameters())
# Gom toàn bộ phần PhoBERT, Linear classifier và các trọng số khác vào nhóm 1
base_classifier_params = [p for n, p in model_entity_aware.named_parameters() if not n.startswith("crf.")]

optimizer_grouped_parameters = [
    {'params': base_classifier_params, 'lr': 3e-5}, # LR nhỏ bảo vệ PhoBERT
    {'params': crf_params, 'lr': 3e-3}             # LR lớn tăng tốc học cho CRF
]
optimizer_entity_aware = AdamW(optimizer_grouped_parameters, weight_decay=0.01)

# 3. Cấu hình Training Arguments chuẩn chỉnh
training_args_aware = TrainingArguments(
    output_dir="./entity_aware_vimedner_model",
    eval_strategy="epoch",
    learning_rate=3e-5, # Giá trị mặc định chung, sẽ bị ghi đè bởi custom optimizer bên trên
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    max_grad_norm=1.0,
    report_to="none"
)

# 4. Khởi tạo Trainer hoàn chỉnh kèm Early Stopping và Custom Optimizer
trainer_entity_aware = Trainer(
    model=model_entity_aware,
    args=training_args_aware,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_entity_aware, None), # Cắm optimizer tách biệt LR vào đây
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)]
)

# 5. Bắt đầu huấn luyện
trainer_entity_aware.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3908/253520766.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('w_class', torch.tensor(static_weights, dtype=torch.float))


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,F1 Nnb
1,No log,12.616293,0.628202,0.638658,0.633387,0.879518,0.007692
2,17.399621,9.506961,0.631859,0.701745,0.664971,0.880034,0.146667
3,17.399621,8.178308,0.710077,0.692349,0.701101,0.896058,0.140244
4,6.706945,8.056319,0.636985,0.725906,0.678545,0.876776,0.225403
5,6.706945,8.568905,0.652784,0.748993,0.697587,0.885749,0.279720
6,4.040214,8.554275,0.688803,0.718389,0.703285,0.892533,0.309859
7,2.769810,9.183544,0.703944,0.728322,0.715926,0.896165,0.312373
8,2.769810,9.915575,0.666348,0.748993,0.705258,0.891785,0.339695
9,1.882101,10.389604,0.681447,0.738523,0.708838,0.893566,0.361158
10,1.882101,10.787138,0.700437,0.731275,0.715524,0.898747,0.339695


TrainOutput(global_step=3718, training_loss=4.788189646375635, metrics={'train_runtime': 2284.5213, 'train_samples_per_second': 30.026, 'train_steps_per_second': 1.878, 'total_flos': 0.0, 'train_loss': 4.788189646375635, 'epoch': 13.0})

In [ ]:
pred_results = trainer_entity_aware.predict(test_dataset)

compute_eval_classify_metrics((pred_results.predictions, pred_results.label_ids))
compute_metrics((pred_results.predictions, pred_results.label_ids))


📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.6763,0.6851,0.6806,308
1,bien_phap_dieu_tri,0.6392,0.6139,0.6263,632
2,nguyen_nhan_benh,0.4045,0.3913,0.3978,276
3,ten_benh,0.8069,0.8694,0.8370,1822
4,trieu_chung_benh,0.6913,0.6573,0.6739,712
5,OVERALL,0.7211,0.7357,0.7284,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7778,0.7727,0.7752,308
1,B-bien_phap_dieu_tri,0.7326,0.6978,0.7147,632
2,B-nguyen_nhan_benh,0.4829,0.4601,0.4712,276
3,B-ten_benh,0.8398,0.9034,0.8704,1822
4,B-trieu_chung_benh,0.7570,0.7177,0.7368,712
5,I-bien_phap_chan_doan,0.7521,0.6074,0.6720,1034
6,I-bien_phap_dieu_tri,0.6769,0.6066,0.6398,1716
7,I-nguyen_nhan_benh,0.5309,0.4231,0.4709,1014
8,I-ten_benh,0.8687,0.9146,0.8910,4448
9,I-trieu_chung_benh,0.7410,0.5543,0.6342,1456


{'precision': np.float64(0.721118661787768),
 'recall': np.float64(0.7357333333333334),
 'f1': np.float64(0.7283526927138331),
 'accuracy': 0.9007458508659455,
 'f1_NNB': np.float64(0.39779005524861877)}

In [ ]:
import os
import torch

# Đường dẫn lưu model trên Drive
save_dir = "/content/drive/MyDrive/ALESRD_1.25"
os.makedirs(save_dir, exist_ok=True)

print("💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...")
tokenizer.save_pretrained(save_dir)

# Lưu toàn bộ state_dict của mô hình (bao gồm cả PhoBERT, Classifier, CRF và Focal Loss)
torch.save(trainer_entity_aware.model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
print(f"✅ Đã lưu mô hình thành công tại: {save_dir}")

💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...
✅ Đã lưu mô hình thành công tại: /content/drive/MyDrive/ALESRD_1.25


In [ ]:
import numpy as np
import os

# Định nghĩa thư mục lưu trên Google Drive (Thành có thể sửa lại tên thư mục nếu muốn)
drive_dir = "/content/drive/MyDrive/dataViMedNER"
os.makedirs(drive_dir, exist_ok=True) # Tự động tạo thư mục nếu chưa có

# Trỏ file thẳng lên Drive
output_file = os.path.join(drive_dir, "testset_error_analysis_ALESRD_1.25.txt")

# 1. Chạy dự đoán trên tập test
print("Đang chạy dự đoán trên tập test...")
pred_results = trainer_entity_aware.predict(test_dataset)

# Lấy .predictions từ kết quả trả về
logits = pred_results.predictions
label_ids = pred_results.label_ids

# Lấy nhãn có xác suất cao nhất
pred_ids = np.argmax(logits, axis=2)

# 2. Trích xuất và định dạng file báo cáo lỗi
print(f"Đang xuất file phân tích lỗi thẳng lên Drive tại:\n{output_file}...")

error_count = 0
with open(output_file, "w", encoding="utf-8") as f:
    # Lặp qua từng câu trong tập test
    for i in range(len(test_dataset)):
        input_ids = test_dataset[i]["input_ids"].tolist()
        true_labels = label_ids[i]
        pred_labels = pred_ids[i]

        # Chuyển ID thành chữ để đọc (Sub-word)
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        sentence_has_error = False
        sentence_log = []

        for token, true_id, pred_id in zip(tokens, true_labels, pred_labels):
            # Bỏ qua các token đặc biệt hoặc sub-token bị đánh dấu -100
            if true_id == -100:
                continue

            true_tag = id2label[true_id]
            pred_tag = id2label[pred_id]

            # So sánh và đánh dấu
            if true_tag != pred_tag:
                sentence_has_error = True
                mark = "❌ SAI"
            else:
                mark = "✅"

            # Xóa dấu '_' của PhoBERT để dễ đọc hơn (tùy chọn)
            clean_token = token.replace("@@", "")

            sentence_log.append(f"{clean_token:<20} | True: {true_tag:<22} | Pred: {pred_tag:<22} | {mark}")

        # CHỈ LƯU NHỮNG CÂU CÓ LỖI SAI ĐỂ DỄ QUAN SÁT
        if sentence_has_error:
            error_count += 1
            f.write(f"================ CÂU SỐ {i} ================\n")
            f.write("\n".join(sentence_log))
            f.write("\n\n")

print(f"✅ Hoàn tất! Tìm thấy {error_count} câu có lỗi sai.")
print(f"Truy cập Google Drive của bạn để xem file: {output_file}")

Đang chạy dự đoán trên tập test...


Đang xuất file phân tích lỗi thẳng lên Drive tại:
/content/drive/MyDrive/dataViMedNER/testset_error_analysis_ALESRD_1.25.txt...
✅ Hoàn tất! Tìm thấy 870 câu có lỗi sai.
Truy cập Google Drive của bạn để xem file: /content/drive/MyDrive/dataViMedNER/testset_error_analysis_ALESRD_1.25.txt


In [ ]:
pred_results = trainer_entity_aware.predict(train_dataset)

predictions = pred_results.predictions
labels = pred_results.label_ids
# ==========================================
# 3. XUẤT KẾT QUẢ DỰ ĐOÁN ĐỂ PHÂN TÍCH LỖI (ERROR ANALYSIS)
# ==========================================
output_file = "/content/drive/MyDrive/datasetViMedNER/train_error_analysis_ALESRD_1.25.txt"

# Sửa lại từ pred_results thành biến predictions và labels đã có sẵn từ vòng lặp trainer.predict()
pred_ids = np.argmax(predictions, axis=2)
true_ids = labels

predicted_tags_per_sentence = []

for prediction, label in zip(pred_ids, true_ids):
    pred_sent = [id2label[int(p)] for p, l in zip(prediction, label) if l != -100]
    predicted_tags_per_sentence.append(pred_sent)

print(f"\n🔍 Đang xuất file phân tích lỗi...")
with open(output_file, "w", encoding="utf-8") as f:
    for idx, (original_sent, predicted_tags) in enumerate(zip(train_dataset.sentences, predicted_tags_per_sentence)):
        if len(original_sent) == len(predicted_tags):
            for (word, true_tag), pred_tag in zip(original_sent, predicted_tags):
                f.write(f"{word}\t{true_tag}\t{pred_tag}\n")
            f.write("\n")
        else:
            print(f"⚠️ Cảnh báo: Lệch số lượng từ ở câu {idx}, bỏ qua xuất câu này...")

print(f"✅ Đã xuất file đối chiếu thành công tại: {output_file}")


🔍 Đang xuất file phân tích lỗi...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 672, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 2346, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 2567, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 2862, bỏ qua xuất câu này...
⚠️ Cảnh báo: Lệch số lượng từ ở câu 3334, bỏ qua xuất câu này...
✅ Đã xuất file đối chiếu thành công tại: /content/drive/MyDrive/datasetViMedNER/train_error_analysis_ALESRD_1.25.txt


In [ ]:
import pandas as pd
from collections import Counter
import os

def extract_entities(tokens, tags):
    """Bóc tách cụm thực thể từ danh sách token và nhãn BIO"""
    entities = []
    current_ent = []
    current_tag = None
    start_idx = -1

    for i, (token, tag) in enumerate(zip(tokens, tags)):
        if tag.startswith('B-'):
            if current_ent:
                entities.append((" ".join(current_ent), current_tag, start_idx, i - 1))
            current_ent = [token]
            current_tag = tag[2:]
            start_idx = i
        elif tag.startswith('I-') and current_tag == tag[2:]:
            current_ent.append(token)
        else:
            if current_ent:
                entities.append((" ".join(current_ent), current_tag, start_idx, i - 1))
                current_ent = []
                current_tag = None
                start_idx = -1

    if current_ent:
        entities.append((" ".join(current_ent), current_tag, start_idx, len(tokens) - 1))
    return entities

def get_overlap(start1, end1, start2, end2):
    """Kiểm tra 2 cụm có đè lên nhau (overlap) không"""
    return max(0, min(end1, end2) - max(start1, start2) + 1)

def analyze_span_errors(file_path, output_excel_path):
    print(f"Đang phân tích file log: {file_path}...")

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    sentences = []
    current_sent = {"tokens": [], "true_tags": [], "pred_tags": []}

    # Đọc file và gom câu
    for line in lines:
        line = line.strip()
        if not line or line.startswith("====="):
            if current_sent["tokens"]:
                sentences.append(current_sent)
                current_sent = {"tokens": [], "true_tags": [], "pred_tags": []}
            continue

        parts = line.split()
        parts = [p.replace("True:", "").replace("Pred:", "").replace("|", "").strip() for p in parts if p not in ["|", "✅", "❌", "SAI"]]

        if len(parts) >= 3:
            current_sent["tokens"].append(parts[0])
            current_sent["true_tags"].append(parts[1])
            current_sent["pred_tags"].append(parts[2])

    # 3 Nhóm lỗi chính
    missed_errors = []
    spurious_errors = []
    mismatch_errors = []

    for sent in sentences:
        true_ents = extract_entities(sent["tokens"], sent["true_tags"])
        pred_ents = extract_entities(sent["tokens"], sent["pred_tags"])

        matched_true = set()
        matched_pred = set()

        # 1. Khớp hoàn toàn
        for i, t_ent in enumerate(true_ents):
            for j, p_ent in enumerate(pred_ents):
                if t_ent == p_ent:
                    matched_true.add(i)
                    matched_pred.add(j)

        # 2. Bị lệch nhãn hoặc ranh giới
        for i, t_ent in enumerate(true_ents):
            if i in matched_true: continue
            for j, p_ent in enumerate(pred_ents):
                if j in matched_pred: continue
                if get_overlap(t_ent[2], t_ent[3], p_ent[2], p_ent[3]) > 0:
                    mismatch_errors.append((t_ent[0], t_ent[1], p_ent[0], p_ent[1]))
                    matched_true.add(i)
                    matched_pred.add(j)

        # 3. Gốc có, Model gán O
        for i, t_ent in enumerate(true_ents):
            if i not in matched_true:
                missed_errors.append((t_ent[0], t_ent[1], "O"))

        # 4. Gốc O, Model gán thực thể
        for j, p_ent in enumerate(pred_ents):
            if j not in matched_pred:
                spurious_errors.append((p_ent[0], "O", p_ent[1]))

    # Đếm tần suất và xuất DataFrame
    def to_dataframe(error_list, col_names):
        counts = Counter(error_list)
        df = pd.DataFrame([(*key, val) for key, val in counts.items()], columns=col_names + ["Tần suất (Count)"])
        return df.sort_values(by="Tần suất (Count)", ascending=False).reset_index(drop=True)

    df_missed = to_dataframe(missed_errors, ["Cụm từ (Text)", "Nhãn gốc (True)", "Nhãn Model (Pred)"])
    df_spurious = to_dataframe(spurious_errors, ["Cụm từ (Text)", "Nhãn gốc (True)", "Nhãn Model (Pred)"])
    df_mismatch = to_dataframe(mismatch_errors, ["Text Gốc", "Nhãn Gốc", "Text Model", "Nhãn Model"])

    # XUẤT THẲNG RA GOOGLE DRIVE
    print(f"Đang xuất báo cáo ra file Excel: {output_excel_path}...")
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        df_missed.to_excel(writer, sheet_name='Goc_Co_Model_O(Missed)', index=False)
        df_spurious.to_excel(writer, sheet_name='Goc_O_Model_Gan(Spurious)', index=False)
        df_mismatch.to_excel(writer, sheet_name='Gan_Lech(Mismatch)', index=False)

    print(f"✅ Hoàn tất! File Excel đã được lưu trực tiếp vào Drive tại:\n{output_excel_path}")

# ==========================================
# THỰC THI VỚI ĐƯỜNG DẪN ĐÃ CHỈ ĐỊNH
# ==========================================
log_path = "/content/drive/MyDrive/datasetViMedNER/train_error_analysis_ALESRD_1.25.txt"
excel_path = "/content/drive/MyDrive/datasetViMedNER/Report_Loi_Data.xlsx"

analyze_span_errors(log_path, excel_path)

Đang phân tích file log: /content/drive/MyDrive/datasetViMedNER/train_error_analysis_ALESRD_1.25.txt...
Đang xuất báo cáo ra file Excel: /content/drive/MyDrive/datasetViMedNER/Report_Loi_Data.xlsx...
✅ Hoàn tất! File Excel đã được lưu trực tiếp vào Drive tại:
/content/drive/MyDrive/datasetViMedNER/Report_Loi_Data.xlsx
